## Make various [element/element] vs. [Fe/H] (probably) plots 

In [1]:


from __future__ import print_function


import matplotlib
matplotlib.use('pdf')
savefig=True

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()

import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import bensby_plotting as bp
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs




print(os.getcwd())

all_wctb
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv
/Users/BenKaiser/Desktop/radial_velocity_calculations


In [2]:
figure_output_dir='/Users/BenKaiser/Desktop/'
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [3]:
lodders_abund_file='Lodders2020_solarsystem_abundances.csv'
solar_system_object_file='20220304_solar_system_body_abundances_select_names.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_all_names.csv'
crust_file='20220304_continental_crust_vals_only.csv'


In [4]:
base_el='Ca'
el_list=['Li','Na','Mg','K','Cr','Fe']
ssp=True

In [5]:
if ssp:
    wd_abund_file='20220304_WD_SSP_abundances.csv'
else:
    wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'



In [6]:
print(os.getcwd())

print(wd_abund_file)
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
crust_table=Table.read(crust_file)
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')

#wd_num_abund_table=Table.read(wd_num_abund_file)
#wd_num_abund_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit




/Users/BenKaiser/Desktop/radial_velocity_calculations
20220304_WD_SSP_abundances.csv


In [7]:
use_indices=np.where(bodies_table['show']==1)
use_bodies_table=bodies_table[use_indices]
#use_wd_indices=np.where((wd_abund_table['show']==1)and (wd_abund_table['show_geo']==1))
use_wd_indices=np.where(wd_abund_table['show_li_evo']==1)
use_wd_abund_table=wd_abund_table[use_wd_indices]
use_wd_indices=np.where(use_wd_abund_table['show']==1)
use_wd_abund_table=use_wd_abund_table[use_wd_indices]

In [8]:

t_step=5

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
met_size=3
ci_size=6
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07

background_alpha=0.4

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

show_all_ssobj_names=False

In [9]:
def plot_wd_errorbar(x_coord, el1el2, el1el2_err,name, selected_marker=wd_marker, markersize=wd_size, label='', color='b'):
    if label=='':
        #label=name #commented this on 2021-11-12 to try to keep star points out of legend
        pass 
    else:
        pass
    uplims=False
    lolims=False
    if np.abs(el1el2_err)> limit_indicator:
        if el1el2_err > 0:
            lolims=True
        elif el1el2_err < 0:
            uplims=True
        else:
            print("This shouldn't print el1el2_err")
        el1el2_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        pass
    plt.errorbar(x_coord, el1el2, yerr= el1el2_err, uplims=uplims, lolims=lolims,  color=color,marker=selected_marker,  markersize=markersize,linestyle='None')
    plt.errorbar(x_coord,el1el2, label=label, marker=selected_marker, markersize=markersize, color=color,linestyle='None')
    return

In [10]:
def plot_CI_chondrite():
    plt.errorbar(0, 0,  color=met_color,marker=met_marker,  markersize=ci_size,linestyle='None')
    return

In [12]:

for i,name in enumerate(el_list):
    spt.initiate_science_plot()
    spt.start_ApJ_fig(width_cols=1,constrained_layout=True,width_height=[0.5,0.5])
    #plt.figure(figsize=(10.,7.25))
    ax=plt.subplot()
    print(i,name)
    num_wds=len(use_wd_abund_table)
    el_string=name.lower()+'/'+base_el.lower()
    #plot_CI_chondrite(start_pos+object_spacing,name,label=CI_label)
    bp.plot_el1el2_FeH(name,base_el,error_bars=False,alpha=background_alpha)
    el1_CI=lodders_table.loc[name]['A_el']
    el2_CI=lodders_table.loc[base_el]['A_el']
    el1_CI_err=lodders_table.loc[name]['A_el_err']
    el2_CI_err=lodders_table.loc[base_el]['A_el_err']
    el1el2_CI=el1_CI-el2_CI
    el1el2_CI_err=np.sqrt(el1_CI_err**2+el2_CI_err**2)
    plot_CI_chondrite()
    if ssp:
        el_string='ssp_'+el_string
        marker=ssp_marker
        markersize=ci_size
    else:
        marker=wd_marker
        markersize=wd_size
    for j,row in enumerate(use_wd_abund_table):
        object_pos=-2.7+(j*0.1)
        el1el2=row[el_string]
        norm_el1el2=el1el2-el1el2_CI
        plt.axhline(y=norm_el1el2,label=row['display_name'],color=row['plot_color'],linestyle='--')
        plot_wd_errorbar(object_pos, norm_el1el2,row[el_string+'_err'],row['display_name'],color=row['plot_color'],selected_marker=marker,markersize=markersize)
        #plt.text(-2.5,norm_el1el2+0.01,row['display_name'],fontsize=figure_text_size)
    plt.ylim(-3.25,2.5)
    #if name!="Li":
        #plt.ylim(-1.1,2.5)
    plt.xlim(-2.75,0.75)
    if savefig:
        print(os.getcwd())
        os.chdir(figure_output_dir)
        print(os.getcwd())
        start = time.time()
        print(start)
        time_string=str(start).split('.')[0]
        if ssp:
            plt.savefig(name+base_el+"_element_evo_ssp"+'_vs_FeH'+'_'+time_string+'.pdf')#plt.grid(True)
        else:
            plt.savefig(name+base_el+"_element_evo"+'_vs_FeH'+'_'+time_string+'.pdf')#plt.grid(True)
        print("Figure saved")
    else:
        pass
    plt.show()
    #plt.text(edge_spot+0.5*el_space,0,name)
#plt.ylabel('log(Z/Ca)')
#plt.xlim()
#plt.ylim(-3.5,2)
#x_ticks=np.arange(0.5*el_space,len(el_list)*el_space,el_space)
#print(x_ticks)
#ax.set_xticks(x_ticks)
#ax.set_xticklabels(el_list)
#plt.legend(loc='best',fontsize=7)
#if savefig:
    #print(os.getcwd())
    #os.chdir(figure_output_dir)
    #print(os.getcwd())
    #start = time.time()
    #print(start)
    #time_string=str(start).split('.')[0]
    #if ssp:
        #plt.savefig("element_evo_ssp"+'_vs_FeH'+'_'+time_string+'.pdf')#plt.grid(True)
    #else:
        #plt.savefig("all_elements"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    #print("Figure saved")
#else:
    #pass
plt.show()

0 Li


****************
KeyError: 'Li/Fe' 
****************


/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1660234375.456032


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


Figure saved
1 Na
/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1660234375.829974


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:50: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


Figure saved
2 Mg
/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1660234376.275595


...

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:50: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1660234377.31445
Figure saved
5 Fe


****************
KeyError: 'Fe/Fe' 
****************


/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1660234377.758734


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:50: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


Figure saved


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:50: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:74: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
